# NiVScan — Step 0: Master Split Dataset

---

| | |
|---|---|
| **Mata Kuliah** | COMP6885001 — Natural Language Processing |
| **Tujuan** | Membuat master split yang digunakan oleh Model B, C, D |
| **Input** | `Dataset_Nipah_Final.csv` (871 baris) + `split_test_manual.csv` (100 kalimat) |
| **Output** | `split_train.csv`, `split_val.csv` |

---

### Prasyarat
**Notebook Model A harus sudah dijalankan terlebih dahulu**, karena file `split_test_manual.csv` dibuat di sana.
Pastikan file berikut sudah ada di Google Drive sebelum lanjut:
- ✅ `Dataset_Nipah_Final.csv` — 887 baris hasil pipeline Model A
- ✅ `split_test_manual.csv` — 179 kalimat ANNOTATED_DATA

### Alur Evaluasi Keempat Model

```
split_test_manual.csv (100 kalimat, label MANUAL)
         │
         └── Evaluasi SEMUA model (A, B, C, D) ← ground truth yang sama

Dataset_Nipah_Final.csv (871 baris) - dikurangi 100 kalimat test
         │
         ├── split_train.csv (~616 baris, 80%) → fine-tuning Model B, C, D
         └── split_val.csv   (~155 baris, 20%) → early stopping Model B, C, D
```

### Notebook ini hanya dijalankan SEKALI oleh satu anggota tim!
**`random_state=42` — jangan diubah!**

---
## 1. Install & Import

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

print('✅ Library berhasil diimport!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

✅ Library berhasil diimport!
   pandas  : 2.2.2
   numpy   : 2.0.2


---
## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil terhubung!')

Mounted at /content/drive
✅ Google Drive berhasil terhubung!


In [3]:
BASE_PATH = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset'

PATH_FINAL       = f'{BASE_PATH}/Dataset_Nipah_Final.csv'
PATH_TEST_MANUAL = f'{BASE_PATH}/split_test_manual.csv'
PATH_TRAIN       = f'{BASE_PATH}/split_train.csv'           # output notebook ini
PATH_VAL         = f'{BASE_PATH}/split_val.csv'             # output notebook ini

print('Path yang digunakan:')
print(f'   Dataset Final    : {PATH_FINAL}')
print(f'   Test Manual      : {PATH_TEST_MANUAL}')
print(f'   Output Train     : {PATH_TRAIN}')
print(f'   Output Val       : {PATH_VAL}')

Path yang digunakan:
   Dataset Final    : /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/Dataset_Nipah_Final.csv
   Test Manual      : /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/split_test_manual.csv
   Output Train     : /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/split_train.csv
   Output Val       : /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/split_val.csv


---
## 3. Load Dataset & Verifikasi Prasyarat

In [4]:
# Cek dua file prasyarat sudah ada
assert os.path.exists(PATH_FINAL), f'File tidak ditemukan: {PATH_FINAL}\n   → Jalankan notebook Model A terlebih dahulu!'
assert os.path.exists(PATH_TEST_MANUAL), f'File tidak ditemukan: {PATH_TEST_MANUAL}\n   → Jalankan notebook Model A terlebih dahulu!'

# Load kedua file
df_final  = pd.read_csv(PATH_FINAL)
df_test   = pd.read_csv(PATH_TEST_MANUAL)

print('Semua prasyarat terpenuhi!')
print(f'   Dataset_Nipah_Final.csv  : {len(df_final)} baris')
print(f'   split_test_manual.csv    : {len(df_test)} kalimat (ground truth manual)')
print()
print('Contoh split_test_manual.csv:')
print(df_test[['id','text_clean','DISEASE','SYMPTOM','LOCATION']].head(3).to_string())

Semua prasyarat terpenuhi!
   Dataset_Nipah_Final.csv  : 897 baris
   split_test_manual.csv    : 179 kalimat (ground truth manual)

Contoh split_test_manual.csv:
   id                                                                                                                    text_clean DISEASE SYMPTOM LOCATION
0   1  Ancaman Virus Nipah Perlu Diwaspadai, Pakar UMSURA Ingatkan Jalur Penularan dan Pencegahan Universitas Muhammadiyah Surabaya   Nipah     NaN      NaN
1   2                                                  Diagnosis Nipah infection can be diagnosed during illness or after recovery.   Nipah     NaN      NaN
2   3                                                            Waspada Virus Nipah: Ancaman Baru d - Rumah Sakit Pusat Pertamina.   Nipah     NaN      NaN


---
## 4. Pisahkan Train+Val dari Dataset Final

100 kalimat `split_test_manual.csv` dikeluarkan dari `Dataset_Nipah_Final.csv` terlebih dahulu
agar tidak ada **data leakage** antara training set dan test set.

In [5]:
# Kalimat yang ada di test manual dikeluarkan dari dataset final
test_texts   = set(df_test['text_clean'].str.strip())
df_trainval  = df_final[~df_final['text_clean'].str.strip().isin(test_texts)].copy()

# Cek berapa yang berhasil dipisahkan
n_overlap = len(df_final) - len(df_trainval)

print(f'Hasil pemisahan:')
print(f'   Dataset_Nipah_Final.csv  : {len(df_final)} baris')
print(f'   Kalimat test yang dibuang: {n_overlap} kalimat')
print(f'   Sisa untuk train + val   : {len(df_trainval)} baris')
print()
if n_overlap < 80:
    print(f'Hanya {n_overlap} kalimat yang cocok (kurang dari 80).')
    print('   Kemungkinan ada perbedaan spasi/karakter antara kedua file.')
    print('   Periksa kembali apakah split_test_manual.csv berasal dari Model A yang sama.')
else:
    print('Pemisahan berhasil — tidak ada data leakage!')

Hasil pemisahan:
   Dataset_Nipah_Final.csv  : 897 baris
   Kalimat test yang dibuang: 179 kalimat
   Sisa untuk train + val   : 718 baris

Pemisahan berhasil — tidak ada data leakage!


---
## 5. Stratified Split → Train (80%) & Val (20%)

In [6]:
# Kolom stratifikasi: 1 = punya entitas, 0 = tidak
df_trainval['has_entity'] = df_trainval[['DISEASE','SYMPTOM','LOCATION']].notna().any(axis=1).astype(int)

# Split train/val dengan random_state=42 — JANGAN DIUBAH
df_train, df_val = train_test_split(
    df_trainval,
    test_size=0.20,
    random_state=42,
    stratify=df_trainval['has_entity']
)

print('Split berhasil!')
print(f'   Train : {len(df_train)} baris')
print(f'   Val   : {len(df_val)} baris')
print(f'   Test  : {len(df_test)} baris (split_test_manual.csv — sudah ada)')
print(f'   Total : {len(df_train) + len(df_val) + len(df_test)} baris')

Split berhasil!
   Train : 574 baris
   Val   : 144 baris
   Test  : 179 baris (split_test_manual.csv — sudah ada)
   Total : 897 baris


In [8]:
# Verifikasi distribusi entity — harus proporsional
df_test_check = df_test.copy()
df_test_check['has_entity'] = df_test_check[['DISEASE','SYMPTOM','LOCATION']].notna().any(axis=1).astype(int)

def cek_distribusi(nama, df_split):
    pct = df_split['has_entity'].mean() * 100
    print(f'   {nama:6s} → {len(df_split):4d} baris | punya entity: {pct:.1f}% | tidak: {100-pct:.1f}%')

print('Verifikasi distribusi entity per split:')
cek_distribusi('Train', df_train)
cek_distribusi('Val',   df_val)
cek_distribusi('Test',  df_test_check)
print()
print('Distribusi konsisten → tidak ada bias split!')

Verifikasi distribusi entity per split:
   Train  →  574 baris | punya entity: 52.8% | tidak: 47.2%
   Val    →  144 baris | punya entity: 52.8% | tidak: 47.2%
   Test   →  179 baris | punya entity: 65.9% | tidak: 34.1%

Distribusi konsisten → tidak ada bias split!


---
## 6. Simpan ke Google Drive

In [9]:
cols_save = ['id', 'text_clean', 'DISEASE', 'SYMPTOM', 'LOCATION']

df_train[cols_save].to_csv(PATH_TRAIN, index=False)
df_val[cols_save].to_csv(PATH_VAL,     index=False)
# split_test_manual.csv tidak perlu disimpan ulang — sudah ada dari Model A

print('File berhasil disimpan ke Google Drive!')
print(f'   split_train.csv → {len(df_train)} baris')
print(f'   split_val.csv   → {len(df_val)} baris')
print(f'   split_test_manual.csv → sudah ada ({len(df_test)} baris)')

File berhasil disimpan ke Google Drive!
   split_train.csv → 574 baris
   split_val.csv   → 144 baris
   split_test_manual.csv → sudah ada (179 baris)


---
## 7. Verifikasi Final — Reload & Cek

In [10]:
check_train = pd.read_csv(PATH_TRAIN)
check_val   = pd.read_csv(PATH_VAL)
check_test  = pd.read_csv(PATH_TEST_MANUAL)

print('Verifikasi final — semua file berhasil dibaca dari Drive:')
print(f'   split_train.csv       : {len(check_train)} baris, kolom: {check_train.columns.tolist()}')
print(f'   split_val.csv         : {len(check_val)} baris, kolom: {check_val.columns.tolist()}')
print(f'   split_test_manual.csv : {len(check_test)} baris, kolom: {check_test.columns.tolist()}')
print()
print('Contoh baris dari split_train.csv:')
print(check_train[['id','text_clean','DISEASE','SYMPTOM','LOCATION']].head(3).to_string())

Verifikasi final — semua file berhasil dibaca dari Drive:
   split_train.csv       : 574 baris, kolom: ['id', 'text_clean', 'DISEASE', 'SYMPTOM', 'LOCATION']
   split_val.csv         : 144 baris, kolom: ['id', 'text_clean', 'DISEASE', 'SYMPTOM', 'LOCATION']
   split_test_manual.csv : 179 baris, kolom: ['id', 'text_clean', 'DISEASE', 'SYMPTOM', 'LOCATION']

Contoh baris dari split_train.csv:
    id                                                                                                                                      text_clean              DISEASE SYMPTOM LOCATION
0  749                                      No infection of humans or other species has been observed in Cambodia, Thailand, or Africa as of May 2018.                  NaN     NaN      NaN
1  861  It is difficult to distinguish Nipah from other infectious diseases, or other causes of encephalitis or pneumonia, without laboratory testing.  Nipah, encephalitis     NaN      NaN
2  329                                 

In [12]:
print('=' * 62)
print('STEP 0 SELESAI — Master Split Berhasil Dibuat!')
print('=' * 62)
print()
print('File yang tersedia di Drive:')
print(f'   split_train.csv       → {len(check_train)} baris  (fine-tuning Model B, C, D)')
print(f'   split_val.csv         → {len(check_val)} baris  (early stopping Model B, C, D)')
print(f'   split_test_manual.csv → {len(check_test)} baris  (evaluasi SEMUA model A, B, C, D)')
print()
print('Langkah selanjutnya untuk masing-masing model:')
print('   Model A → hasil evaluasi sudah ada di notebook Model A')
print('   Model B → load split_train + split_val untuk fine-tuning')
print('             evaluasi akhir di split_test_manual.csv')
print('   Model C → sama seperti Model B')
print('   Model D → sama seperti Model B')
print()
print('JANGAN jalankan notebook ini lagi!')
print('   random_state=42 sudah mengunci split ini.')

STEP 0 SELESAI — Master Split Berhasil Dibuat!

File yang tersedia di Drive:
   split_train.csv       → 574 baris  (fine-tuning Model B, C, D)
   split_val.csv         → 144 baris  (early stopping Model B, C, D)
   split_test_manual.csv → 179 baris  (evaluasi SEMUA model A, B, C, D)

Langkah selanjutnya untuk masing-masing model:
   Model A → hasil evaluasi sudah ada di notebook Model A
   Model B → load split_train + split_val untuk fine-tuning
             evaluasi akhir di split_test_manual.csv
   Model C → sama seperti Model B
   Model D → sama seperti Model B

JANGAN jalankan notebook ini lagi!
   random_state=42 sudah mengunci split ini.
